# Module 1 Baseline: NYC TLC Green Taxi Trip Duration

This notebook downloads one month of NYC TLC green taxi trip records in Parquet format, engineers `PU_DO` and `trip_distance`, trains a `DictVectorizer` + `LinearRegression` baseline, reports validation RMSE and MAE, and saves the fitted artifacts to `models/baseline.pkl`.


In [ ]:
from pathlib import Path
import pickle
import urllib.request

import numpy as np
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data"
MODEL_DIR = ROOT / "models"
REPORT_PATH = ROOT / "reports" / "module-1.md"
DATA_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)
REPORT_PATH.parent.mkdir(exist_ok=True)

YEAR = 2024
MONTH = 1
DATA_URL = f"https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_{YEAR}-{MONTH:02d}.parquet"
DATA_PATH = DATA_DIR / f"green_tripdata_{YEAR}-{MONTH:02d}.parquet"
MODEL_PATH = MODEL_DIR / "model.pkl"

print(f"Data URL: {DATA_URL}")
print(f"Data path: {DATA_PATH}")


Data URL: https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2024-01.parquet
Data path: /home/ahmed/projects/mlops_mini_projects/session_1_mini_project/s1_mini_project/data/green_tripdata_2024-01.parquet


In [2]:
if not DATA_PATH.exists():
    print("Downloading TLC Parquet file...")
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
else:
    print("Using existing downloaded file.")

print(f"Downloaded size: {DATA_PATH.stat().st_size / 1024**2:.1f} MB")


Using existing downloaded file.
Downloaded size: 1.3 MB


In [3]:
columns = [
    "lpep_pickup_datetime",
    "lpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance",
]

df = pd.read_parquet(DATA_PATH, columns=columns)
df.head()


,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,trip_distance
0,2024-01-01 00:46:55,2024-01-01 00:58:25,236,239,1.98
1,2024-01-01 00:31:42,2024-01-01 00:52:34,65,170,6.54
2,2024-01-01 00:30:21,2024-01-01 00:49:23,74,262,3.08
3,2024-01-01 00:30:20,2024-01-01 00:42:12,74,116,2.40
4,2024-01-01 00:32:38,2024-01-01 00:43:37,74,243,5.14


In [4]:
pickup = pd.to_datetime(df["lpep_pickup_datetime"])
dropoff = pd.to_datetime(df["lpep_dropoff_datetime"])
df["duration"] = (dropoff - pickup).dt.total_seconds() / 60

df = df.rename(columns={
    "PULocationID": "PUlocationID",
    "DOLocationID": "DOlocationID",
})
df["PU_DO"] = df["PUlocationID"].astype("Int64").astype(str) + "_" + df["DOlocationID"].astype("Int64").astype(str)

# Remove malformed records before training.
df = df[
    df["duration"].between(1, 60)
    & df["trip_distance"].between(0.01, 100)
    & df["PUlocationID"].notna()

    & df["DOlocationID"].notna()

].copy()

features = ["PU_DO", "trip_distance"]
target = "duration"
print(f"Rows after filtering: {len(df):,}")
print(df[[*features, target]].describe(include="all"))


Rows after filtering: 52,529
        PU_DO  trip_distance      duration
count   52529   52529.000000  52529.000000
unique   4416            NaN           NaN
top     74_75            NaN           NaN
freq     1864            NaN           NaN
mean      NaN       2.710177     13.541208
std       NaN       2.658869      8.695926
min       NaN       0.010000      1.000000
25%       NaN       1.210000      7.550000
50%       NaN       1.880000     11.466667
75%       NaN       3.170000     17.000000
max       NaN      59.890000     60.000000


In [5]:
# Chronological validation split: the last 20% of trips is validation data.
split_index = int(len(df) * 0.8)
train_df = df.iloc[:split_index].copy()
valid_df = df.iloc[split_index:].copy()

def to_dicts(frame: pd.DataFrame) -> list[dict]:
    return frame[["PU_DO", "trip_distance"]].to_dict(orient="records")

X_train_dicts = to_dicts(train_df)
X_valid_dicts = to_dicts(valid_df)
y_train = train_df[target].to_numpy()
y_valid = valid_df[target].to_numpy()

dv = DictVectorizer()
X_train = dv.fit_transform(X_train_dicts)
X_valid = dv.transform(X_valid_dicts)

model = LinearRegression()
model.fit(X_train, y_train)

predictions = model.predict(X_valid)
rmse = float(np.sqrt(mean_squared_error(y_valid, predictions)))
mae = float(mean_absolute_error(y_valid, predictions))

print(f"Validation RMSE: {rmse:.4f} minutes")
print(f"Validation MAE: {mae:.4f} minutes")


Validation RMSE: 6.4647 minutes
Validation MAE: 3.8969 minutes


In [6]:
artifacts = {
    "model": model,
    "dv": dv,
    "features": features,
    "target": target,
    "data_url": DATA_URL,
    "year": YEAR,
    "month": MONTH,
    "metrics": {"rmse": rmse, "mae": mae},
}

with MODEL_PATH.open("wb") as f_out:
    pickle.dump(artifacts, f_out)

print(f"Saved fitted model artifacts to {MODEL_PATH}")


Saved fitted model artifacts to /home/ahmed/projects/mlops_mini_projects/session_1_mini_project/s1_mini_project/models/baseline.pkl


## Baseline results

The validation metrics printed above are written to `reports/module-1.md` by the final cell.


In [7]:
report_text = f"""# Module 1 Baseline\n\n## Validation metrics\n\n- **Dataset:** NYC TLC green taxi trips, {YEAR}-{MONTH:02d}\n- **Features:** `PU_DO`, `trip_distance`\n- **Model:** `DictVectorizer` + `LinearRegression`\n- **Validation split:** chronological final 20% of filtered records\n- **Validation RMSE:** {rmse:.4f} minutes\n- **Validation MAE:** {mae:.4f} minutes\n\n## Reproduction\n\nRun `notebooks/00-baseline.ipynb` to download the official Parquet file, reproduce the metrics, and overwrite `models/baseline.pkl`.\n"""

REPORT_PATH.write_text(report_text, encoding="utf-8")
print(REPORT_PATH.read_text())


# Module 1 Baseline

## Validation metrics

- **Dataset:** NYC TLC green taxi trips, 2024-01
- **Features:** `PU_DO`, `trip_distance`
- **Model:** `DictVectorizer` + `LinearRegression`
- **Validation split:** chronological final 20% of filtered records
- **Validation RMSE:** 6.4647 minutes
- **Validation MAE:** 3.8969 minutes

## Reproduction

Run `notebooks/00-baseline.ipynb` to download the official Parquet file, reproduce the metrics, and overwrite `models/baseline.pkl`.

